# 08 · Final experiment: public U-Net with RGB and RGB+NIR

Adapt the WorldFloods **WFV1_unet** architecture and published training settings to Sen1Floods11. Train two binary segmentation models from scratch, comparing B4/B3/B2 with B4/B3/B2/B8. This is the experiment used in the project Summary; both completed cloud runs selected **epoch 18**.

This notebook runs independently of 01–07 on the configured CUDA environment. It retains the complete training, evaluation and export code. Returned results are presented in [the English Summary](00_project_summary_en.ipynb).

Sources: [published configuration](https://spaceml-org.github.io/ml4floods/content/ml4ops/HOWTO_Run_Inference_on_new_data.html), [pinned U-Net implementation](https://github.com/spaceml-org/ml4floods/blob/f21129d1de0786eddb6b95118ccc47598fc3c40b/ml4floods/models/architectures/unets.py), and [loss implementation](https://github.com/spaceml-org/ml4floods/blob/f21129d1de0786eddb6b95118ccc47598fc3c40b/ml4floods/models/utils/losses.py).

## 1. Fixed experimental configuration

| Setting | Value and rationale |
|---|---|
| Architecture | U-Net widths 64/128/256/512; three pools; bilinear decoder |
| Input / output | 3 or 4 input bands; 2 logits per pixel |
| Split | Official train 252 / validation 89 / test 90 / Bolivia 15 |
| Training samples | Four non-overlapping 256 × 256 windows per training image |
| Budget | 25 epochs, batch size 32; published configuration |
| Optimizer | Adam, learning rate 1e-4, weight decay 0 |
| Scheduler | ReduceLROnPlateau on validation Dice loss; factor 0.5, patience 2 |
| Objective | 0.5 weighted CE + 0.5 Dice; Dice smoothing = 1 |
| Class weights | Training valid-pixel counts: N/n_c |
| Standardization | Training-derived per-band mean and population standard deviation |
| Initialization | Seed 12; matching layers share initial weights |
| Augmentation | No extra flips or random crops |
| Runtime | CUDA AMP for the forward pass; FP32 loss; num_workers=0 |
| Selection | Minimum validation Dice loss; no early stopping |

Each epoch uses **1008 windows and 32 batches**, for up to 800 optimizer updates per model. Evaluation uses full 512 × 512 images. The full defaults and adaptations are stored in `config.json`.

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import json
import time
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from IPython.display import display, Markdown
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
BASE = ROOT / "data/sen1floods11/v1.1"
HAND = BASE / "data/flood_events/HandLabeled"
CFG = dict(epochs=25, batch_size=32, crop_size=256, lr=1e-4,
           weight_decay=0.0, seed=12, num_workers=0, amp=True,
           loss="0.5_inverse_frequency_CE_plus_0.5_valid_masked_Dice", dice_smooth=1.0,
           input_standardization="training_valid_pixel_mean_std",
           architecture="worldfloods_v1_unet_64_128_256_512_bilinear",
           upstream_revision="f21129d1de0786eddb6b95118ccc47598fc3c40b",
           published_config="WFV1_unet", train_windows="four_nonoverlapping_256_per_512",
           augmentation="none", class_weight_formula="N/n_c",
           adam_betas=[0.9, 0.999], adam_eps=1e-8, early_stopping=False,
           monitor="valid_dice_loss", scheduler_factor=0.5, scheduler_patience=2,
           scheduler_threshold=1e-4, scheduler_threshold_mode="rel",
           scheduler_cooldown=0, scheduler_min_lr=0.0, scheduler_eps=1e-8,
           dice_ignore_adaptation="mask_both_prediction_and_one_hot_exclude_empty_images")
EXPERIMENTS = {"rgb": ["B4", "B3", "B2"],
               "rgb_nir": ["B4", "B3", "B2", "B8"]}
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
USE_AMP = CFG["amp"] and device.type == "cuda"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
RUN = ROOT / "runs" / datetime.now().strftime("worldfloods_unet_%Y%m%d_%H%M%S_%f")
RUN.mkdir(parents=True)
(RUN / "config.json").write_text(json.dumps({"training": CFG, "bands": EXPERIMENTS,
    "torch": str(torch.__version__), "device": str(device)}, indent=2), encoding="utf-8")
print("Results:", RUN)

splits = {}
for split in ["train", "valid", "test", "bolivia"]:
    with (BASE / f"splits/flood_handlabeled/flood_{split}_data.csv").open(newline="") as f:
        splits[split] = list(csv.reader(f))
print("Official split sizes:", {key: len(rows) for key, rows in splits.items()})

## 2. Fixed windows and shared valid pixels

Use the official image lists and a shared B4/B3/B2/B8 validity mask. Convert training DN to reflectance with DN/10000, then standardize using training statistics. Missing inputs become zero; invalid targets remain -1.

Read four non-overlapping 256 × 256 windows from each training image. Both models use the same shuffled window order. Validation, testing and statistics use full images.

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, rows, channels, training=False, normalize=True):
        self.rows = rows
        self.channels = channels
        self.training = training
        self.epoch = 0
        self.normalize = normalize

    def __len__(self):
        return len(self.rows) * (4 if self.training else 1)

    def __getitem__(self, index):
        row_index = index // 4 if self.training else index
        window = None
        if self.training:
            tile = index % 4
            size = CFG["crop_size"]
            window = rasterio.windows.Window((tile % 2) * size, (tile // 2) * size, size, size)
        image_name, label_name = self.rows[row_index]
        image_name = image_name.replace("_S1Hand", "_S2Hand")
        with rasterio.open(HAND / "S2Hand" / image_name) as src:
            bands = [src.descriptions.index(b) + 1 for b in ["B4", "B3", "B2", "B8"]]
            x = src.read(bands, window=window, masked=True).astype("float32").filled(np.nan)
        with rasterio.open(HAND / "LabelHand" / label_name) as src:
            y = src.read(1, window=window).astype("int64")
        y[~np.isfinite(x).all(axis=0)] = -1
        x = x / 10000.0
        if self.normalize:
            x = (x - CHANNEL_MEAN[:, None, None]) / CHANNEL_STD[:, None, None]
        x = np.nan_to_num(x)
        return torch.from_numpy(x[:self.channels].copy()), torch.from_numpy(y.copy())

### Training-derived statistics

Scan all 252 full training images. Accumulate means and population standard deviations in float64 over pixels valid in all four bands and in the labels. Save full-precision values for later inference.

Class weights are **N/n_c**, approximately 1.1051 for non-water and 10.5149 for water. Weighted CE is divided by the number of valid pixels, matching the selected public loss convention. These statistics come only from training data.

In [ ]:
sums = np.zeros(4, dtype=np.float64)
squares = np.zeros(4, dtype=np.float64)
class_counts = np.zeros(2, dtype=np.int64)
raw_dataset = FloodDataset(splits["train"], channels=4, normalize=False)
for index in range(len(raw_dataset)):
    x_raw, label = raw_dataset[index]
    keep = label.numpy() != -1
    values = x_raw.numpy()[:, keep].astype(np.float64)
    sums += values.sum(axis=1)
    squares += np.square(values).sum(axis=1)
    class_counts += np.bincount(label.numpy()[keep], minlength=2)
count = int(class_counts.sum())
means = sums / count
stds = np.sqrt(squares / count - means**2)
CHANNEL_MEAN = means.astype(np.float32)
CHANNEL_STD = stds.astype(np.float32)
weights = class_counts.sum() / class_counts
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32, device=device)
CFG["class_counts"] = class_counts.tolist()
CFG["class_weights"] = weights.tolist()
CFG["channel_mean"] = CHANNEL_MEAN.tolist()
CFG["channel_std"] = CHANNEL_STD.tolist()
CFG["statistics_bands"] = ["B4", "B3", "B2", "B8"]
(RUN / "config.json").write_text(json.dumps({"training": CFG, "bands": EXPERIMENTS,
    "torch": str(torch.__version__), "device": str(device)}, indent=2), encoding="utf-8")
print("[Non-water, water] class counts:", class_counts)
print("Class weights:", weights)
print("[B4, B3, B2, B8] mean:", CHANNEL_MEAN)
print("[B4, B3, B2, B8] std:", CHANNEL_STD)

## 3. U-Net and common initialization

Use two padded 3 × 3 convolutions with ReLU per block, encoder widths 64→128→256→512 and decoder widths 256→128→64. Bilinear upsampling uses `align_corners=True`, with skip concatenation and a two-class 1 × 1 output head.

Both models share initial values for matching layers; RGB takes the first three channels of the reference input layer. The small forward check below verifies the interface before training.

In [ ]:
import torch
from torch import nn

class DoubleConv(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.encoders = nn.ModuleList([
            DoubleConv(in_channels, 64), DoubleConv(64, 128),
            DoubleConv(128, 256), DoubleConv(256, 512),
        ])
        self.pool = nn.MaxPool2d(2)
        self.decoders = nn.ModuleList([
            DoubleConv(512 + 256, 256), DoubleConv(256 + 128, 128),
            DoubleConv(128 + 64, 64),
        ])
        self.head = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        skips = []
        for i, encoder in enumerate(self.encoders):
            if i:
                x = self.pool(x)
            x = encoder(x)
            skips.append(x)
        for decoder, skip in zip(self.decoders, reversed(skips[:-1])):
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=True)
            x = decoder(torch.cat([x, skip], dim=1))
        return self.head(x)


torch.manual_seed(CFG["seed"])
reference = UNet(in_channels=4, num_classes=2)
initial_state = {key: value.detach().clone() for key, value in reference.state_dict().items()}
del reference

def create_model(channels):
    model = UNet(in_channels=channels, num_classes=2)
    state = dict(initial_state)
    state["encoders.0.0.weight"] = initial_state["encoders.0.0.weight"][:, :channels].clone()
    model.load_state_dict(state)
    return model.to(device)

for channels in [3, 4]:
    preview = create_model(channels).eval()
    with torch.inference_mode():
        output = preview(torch.zeros(1, channels, 32, 32, device=device))
    print(f"{channels} bands: parameters={sum(p.numel() for p in preview.parameters()):,}; "
          f"preview (1,{channels},32,32) -> {tuple(output.shape)}")
    del preview, output

## 4. Loss and evaluation

The objective is **0.5 CE + 0.5 Dice**. CE multiplies each valid pixel by its training class weight and divides by the valid-pixel count. Dice is computed per image and class, with smoothing 1, then averaged across valid images and both classes.

For this dataset adaptation, ignored labels are masked in both probabilities and one-hot targets; all-ignored images are excluded. This differs from the pinned upstream handling of ignored target pixels. Forward computation can use AMP; loss reductions use FP32.

Validation Dice loss drives scheduling and checkpoint selection. Water IoU = TP/(TP+FP+FN), precision = TP/(TP+FP), recall = TP/(TP+FN), and F1 = 2TP/(2TP+FP+FN). Global IoU pools valid pixels; mean IoU averages image-level water IoUs, excluding images with zero water union. Undefined ratios remain NaN. Evaluation metrics are unweighted.

In [ ]:
def divide(numerator, denominator):
    return float(numerator / denominator) if denominator else float("nan")

class WaterMetrics:
    def __init__(self):
        self.matrix = np.zeros((2, 2), dtype=np.int64)
        self.image_ious = []
        self.valid_images = 0

    def update(self, prediction, target):
        for pred, truth in zip(prediction.cpu(), target.cpu()):
            valid = truth != -1
            if not valid.any():
                continue
            matrix = torch.bincount(2 * truth[valid] + pred[valid], minlength=4).reshape(2, 2).numpy()
            self.matrix += matrix
            self.valid_images += 1
            tn, fp, fn, tp = matrix.ravel()
            union = tp + fp + fn
            if union:
                self.image_ious.append(float(tp / union))

    def compute(self):
        tn, fp, fn, tp = self.matrix.ravel()
        return dict(global_water_iou=divide(tp, tp + fp + fn),
                    mean_water_iou=float(np.mean(self.image_ious)) if self.image_ious else float("nan"),
                    precision=divide(tp, tp + fp), recall=divide(tp, tp + fn),
                    f1=divide(2 * tp, 2 * tp + fp + fn),
                    true_water_fraction=divide(tp + fn, self.matrix.sum()),
                    predicted_water_fraction=divide(tp + fp, self.matrix.sum()),
                    valid_pixels=int(self.matrix.sum()), valid_images=self.valid_images,
                    images_with_water_union=len(self.image_ious))

def segmentation_loss(logits, target):
    logits = logits.float()
    valid = target != -1
    ce = F.cross_entropy(logits, target, weight=CLASS_WEIGHTS,
                         ignore_index=-1, reduction="sum") / valid.sum()
    mask = valid.unsqueeze(1)
    probability = logits.softmax(dim=1) * mask
    truth = F.one_hot(target.clamp_min(0), num_classes=2).permute(0, 3, 1, 2).float() * mask
    intersection = (probability * truth).sum(dim=(2, 3))
    total = probability.sum(dim=(2, 3)) + truth.sum(dim=(2, 3))
    per_image_class = 1 - (2 * intersection + CFG["dice_smooth"]) / (total + CFG["dice_smooth"])
    dice = per_image_class[valid.flatten(1).any(dim=1)].mean()
    return (ce + dice) * 0.5, ce, dice


def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    metrics = WaterMetrics()
    ce_sum, dice_sum, pixel_count, image_count, skipped = 0.0, 0.0, 0, 0, 0
    optimizer_updates, amp_skipped_steps = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        count = int((y != -1).sum())
        images = int((y != -1).flatten(1).any(dim=1).sum())
        if count == 0:
            skipped += 1
            continue
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
            loss, ce, dice = segmentation_loss(logits, y)
            if training:
                scaler.scale(loss).backward()
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                was_skipped = scaler.get_scale() < previous_scale
                amp_skipped_steps += int(was_skipped)
                optimizer_updates += int(not was_skipped)
        ce_sum += ce.detach().item() * count
        dice_sum += dice.detach().item() * images
        pixel_count += count
        image_count += images
        metrics.update(logits.detach().argmax(dim=1), y)
    result = metrics.compute()
    ce_loss, dice_loss = divide(ce_sum, pixel_count), divide(dice_sum, image_count)
    result.update(loss=(ce_loss + dice_loss) * 0.5, ce_loss=ce_loss, dice_loss=dice_loss,
                  skipped_batches=skipped, optimizer_updates=optimizer_updates,
                  amp_skipped_steps=amp_skipped_steps)
    return result

## 5. Training, scheduling and checkpointing

Train each model for 25 epochs. Each scheduler independently responds to its model's validation Dice loss under the same rule. Save `best.pt` at the lowest validation Dice loss and `last.pt` every epoch.

Histories record CE, Dice, total loss, IoU, water fractions, learning rates and optimizer/AMP update counts. Checkpoints include preprocessing statistics and configuration, but no optimizer state for resuming training.

In [ ]:
def write_csv(path, rows):
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def train_experiment(name, bands):
    out = RUN / name
    out.mkdir(exist_ok=True)
    dataset = FloodDataset(splits["train"], len(bands), training=True)
    train_loader = DataLoader(dataset, batch_size=CFG["batch_size"], shuffle=True,
        num_workers=CFG["num_workers"], generator=torch.Generator().manual_seed(CFG["seed"]))
    valid_loader = DataLoader(FloodDataset(splits["valid"], len(bands)), batch_size=1,
        num_workers=CFG["num_workers"])
    model = create_model(len(bands))
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"],
                                 betas=tuple(CFG["adam_betas"]), eps=CFG["adam_eps"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=CFG["scheduler_factor"], patience=CFG["scheduler_patience"],
        threshold=CFG["scheduler_threshold"], threshold_mode=CFG["scheduler_threshold_mode"],
        cooldown=CFG["scheduler_cooldown"], min_lr=CFG["scheduler_min_lr"], eps=CFG["scheduler_eps"])
    print(name, "training images:", len(splits["train"]), "windows:", len(dataset),
          "batches per epoch:", len(train_loader))
    best_dice, history = float("inf"), []
    for epoch in range(1, CFG["epochs"] + 1):
        start = time.perf_counter()
        current_lr = optimizer.param_groups[0]["lr"]
        train = run_epoch(model, train_loader, optimizer, scaler)
        valid = run_epoch(model, valid_loader)
        scheduler.step(valid["dice_loss"])
        row = {"epoch": epoch, "lr": current_lr, "next_lr": optimizer.param_groups[0]["lr"], "seconds": time.perf_counter() - start,
               **{"train_" + k: v for k, v in train.items()},
               **{"valid_" + k: v for k, v in valid.items()}}
        history.append(row)
        write_csv(out / "history.csv", history)
        checkpoint = {"model": model.state_dict(), "bands": bands, "config": CFG,
                      "epoch": epoch, "validation": valid}
        torch.save(checkpoint, out / "last.pt")
        if valid["dice_loss"] < best_dice:
            best_dice = valid["dice_loss"]
            torch.save(checkpoint, out / "best.pt")
        del checkpoint
        print(f"{name} | {epoch:02d}/{CFG['epochs']} | train loss {train['loss']:.4f} | "
              f"lr {current_lr:.2g} | val Dice loss {valid['dice_loss']:.4f} | val mean IoU {valid['mean_water_iou']:.4f} | "
              f"val predicted water {valid['predicted_water_fraction']:.2%} | "
              f"{row['seconds']:.0f}s", flush=True)
    del model, optimizer, scaler, scheduler
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return history

## 6. Train the two models

Execute RGB, then RGB+NIR, under a new `runs/worldfloods_unet_*` directory. This cell performs training; the following sections generate curves, test metrics and prediction exports.

In [ ]:
histories = {}
for name, bands in EXPERIMENTS.items():
    histories[name] = train_experiment(name, bands)
print("Training finished:", RUN)

for name, history in histories.items():
    print(name, "optimizer updates:", sum(r["train_optimizer_updates"] for r in history),
          "AMP skipped steps:", sum(r["train_amp_skipped_steps"] for r in history))


## 7. Learning curves

Solid and dashed lines show training and validation loss. The second panel shows validation image-mean water IoU. Values are read directly from this run's epoch logs.

The plotted loss is 0.5 CE + 0.5 Dice. Checkpoints are selected by validation Dice loss, not by the displayed IoU.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for (name, history), color in zip(histories.items(), ["#167caf", "#df893b"]):
    epochs = [row["epoch"] for row in history]
    axes[0].plot(epochs, [row["train_loss"] for row in history], color=color, label=name + " train")
    axes[0].plot(epochs, [row["valid_loss"] for row in history], color=color, linestyle="--", label=name + " valid")
    axes[1].plot(epochs, [row["valid_mean_water_iou"] for row in history], color=color, label=name)
axes[0].set(xlabel="Epoch", ylabel="0.5 CE + 0.5 Dice", title="Training and validation loss")
axes[1].set(xlabel="Epoch", ylabel="Image-mean water IoU", title="Validation water IoU", ylim=(0, 1))
for ax in axes:
    ax.legend(); ax.grid(alpha=0.2)
fig.savefig(RUN / "learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

## 8. Test evaluation

Load each minimum-validation-Dice `best.pt` and evaluate the 90 test images and 15 Bolivia images separately. Save `test_metrics.csv`. Predictions use argmax, with ties assigned to non-water; test scores do not select checkpoints.

These fixed test splits were also inspected in earlier experiments; results describe this single-seed comparison.

In [ ]:
def load_best(name):
    checkpoint = torch.load(RUN / name / "best.pt", map_location="cpu", weights_only=True)
    model = UNet(in_channels=len(checkpoint["bands"]), num_classes=2)
    model.load_state_dict(checkpoint["model"])
    return model.to(device).eval(), checkpoint

results = []
for name, bands in EXPERIMENTS.items():
    model, checkpoint = load_best(name)
    for split in ["test", "bolivia"]:
        loader = DataLoader(FloodDataset(splits[split], len(bands)), batch_size=1,
                            num_workers=CFG["num_workers"])
        metrics = run_epoch(model, loader)
        results.append({"experiment": name, "split": split,
                        "best_epoch": checkpoint["epoch"], **metrics})
    del model, checkpoint
write_csv(RUN / "test_metrics.csv", results)
columns = ["experiment", "split", "best_epoch", "mean_water_iou", "global_water_iou", "f1", "precision", "recall"]
lines = ["| " + " | ".join(columns) + " |", "|" + "---|" * len(columns)]
for row in results:
    lines.append("| " + " | ".join(f"{row[k]:.4f}" if isinstance(row[k], float) else str(row[k]) for k in columns) + " |")
display(Markdown("\n".join(lines)))


## 9. Prediction example

Compare both models on the first image in the official test list (`SHOW_INDEX=0`). Display RGB, NIR, human labels and both predictions on the same grid. RGB stretching is for display only; NIR uses a fixed 0–0.6 range. Grey, beige and blue indicate ignored, non-water and water pixels.

Export probabilities and masks as georeferenced TIFFs with nodata=-1. Model inputs retain the experiment's fixed preprocessing.

In [ ]:
SHOW_INDEX = 0
x4, target = FloodDataset(splits["test"], 4)[SHOW_INDEX]
image_name = splits["test"][SHOW_INDEX][0].replace("_S1Hand", "_S2Hand")
with rasterio.open(HAND / "S2Hand" / image_name) as src:
    profile = src.profile.copy()
valid = target.numpy() != -1
predictions = {}
for name, bands in EXPERIMENTS.items():
    model, checkpoint = load_best(name)
    with torch.inference_mode():
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x4[:len(bands)].unsqueeze(0).to(device))
        probability = logits.float().softmax(dim=1)[0, 1].cpu().numpy()
        mask = logits.argmax(dim=1)[0].cpu().numpy().astype("int16")
    mask[~valid] = -1
    probability[~valid] = -1
    predictions[name] = mask
    for suffix, array, dtype in [("mask", mask, "int16"), ("probability", probability, "float32")]:
        output_profile = {**profile, "count": 1, "dtype": dtype, "nodata": -1, "compress": "deflate"}
        with rasterio.open(RUN / name / f"{Path(image_name).stem}_{suffix}.tif", "w", **output_profile) as dst:
            dst.write(array, 1)
    del model, checkpoint
display_x4, _ = FloodDataset(splits["test"], 4, normalize=False)[SHOW_INDEX]
rgb = np.moveaxis(display_x4[:3].numpy(), 0, -1)
lo, hi = np.percentile(rgb[valid], [2, 98], axis=0)
rgb = np.clip((rgb - lo) / np.maximum(hi - lo, 1e-6), 0, 1)
cmap = ListedColormap(["#d7dee7", "#eee8d5", "#167caf"])
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], 3)
fig, axes = plt.subplots(1, 5, figsize=(17, 4), layout="constrained")
axes[0].imshow(rgb); axes[0].set_title("RGB")
axes[1].imshow(display_x4[3], cmap="gray", vmin=0, vmax=0.6); axes[1].set_title("NIR · B8")
for ax, values, title in zip(axes[2:], [target, predictions["rgb"], predictions["rgb_nir"]],
                            ["Human label", "RGB prediction", "RGB+NIR prediction"]):
    im = ax.imshow(values, cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_title(title)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
bar = fig.colorbar(im, ax=list(axes[2:]), ticks=[-1, 0, 1], shrink=0.6)
bar.ax.set_yticklabels(["Ignored", "Non-water", "Water"])
fig.suptitle(Path(image_name).stem)
fig.savefig(RUN / "test_prediction_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)


## Outputs

The run directory contains configuration, checkpoints, epoch CSVs, learning curves, test metrics and the prediction example. Save the executed notebook alongside these files.

References: [U-Net](https://arxiv.org/abs/1505.04597) · [Sen1Floods11](https://github.com/cloudtostreet/Sen1Floods11).

The final model adapts the published method to three/four-band inputs, two classes, training-derived statistics and valid-pixel Dice. See 09 for error analysis and 11–12 for the NYC application.